In [1]:
# ============================================================
# 028_config_and_state.ipynb
# ============================================================
#
# Overview
# ----------------
# This notebook acts as the execution bootstrap and state layer for the
# Daily researchOS pipeline.
#
# It is responsible for preparing environment variables, loading and
# normalizing configuration, initializing run context (run_id, timestamps),
# and managing persistent execution state (JSON-based caches and metadata).
#
# This notebook deliberately DOES NOT implement any Notion API I/O.
# All Notion REST calls, retries, CRUD wrappers, and schema introspection
# are delegated to 029_notion_clients_and_io.ipynb.
#
# The primary consumer of this notebook is 036_daily_orchestrator.ipynb,
# which assumes that environment, state, logging, and preflight checks
# have already been completed here.
#
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - env.txt
#       - API keys (NOTION_TOKEN / NOTION_API_KEY, OPENAI_API_KEY, etc.)
#       - Database IDs (Papers / Events / Monitoring DBs)
#   - config.yaml (optional)
#       - Pipeline cadence, lookback windows, thresholds, feature flags
#   - ./state/*.json
#       - Persistent state from prior runs (processed IDs, cursors, metadata)
#
# Outputs:
#   - run_id
#       - Unique execution identifier (ISO timestamp–based)
#   - normalized config dict
#       - Merged and validated configuration (env + yaml)
#   - JSON state files under ./state/
#       - run metadata, caches, cursors, execution markers
#   - run_id-aware logger
#       - Structured logging shared by downstream notebooks
#
#   NOTE:
#   - This notebook exports NO Notion clients or CRUD functions.
#   - If a function talks to Notion, it belongs in 029, not here.
#
#
# Structure
# ----------------
# Cell 01: Imports and dependency setup
# Cell 02: Environment variable loading and validation
# Cell 03: Optional config.yaml loading and normalization
# Cell 04: Run context initialization (run_id, timestamps, execution window)
# Cell 05: State directory setup and verification
# Cell 06: State persistence helpers (atomic JSON read/write)
# Cell 07: Logging configuration with run_id injection
# Cell 08: Lightweight Notion preflight checks via 029
#          (authentication + database accessibility only)
# Cell 09: Exported context helpers for downstream notebooks
# Cell 10: Configuration summary and sanity checks
# Cell 11: RunContext manager for encapsulated execution
# Cell 12: Self-test and validation suite
#
#
# Notes
# ----------------
# - 028 is intentionally boring and defensive.
#   Its job is to make downstream execution predictable and idempotent.
#
# - All Notion API interactions (auth validation, retries, CRUD, schema
#   introspection) must be implemented in 029_notion_clients_and_io.ipynb.
#
# - If you find yourself wanting to call:
#       - notion_get / notion_post / notion_patch
#       - create_page / update_page / query_database
#   from this notebook, that logic belongs in 029 instead.
#
# - State is persisted as transparent JSON files under ./state/ for
#   debuggability and auditability.
#
# - Run IDs are generated once per execution and reused across all
#   downstream notebooks for correlation.
#
# - Preflight checks are intentionally lightweight:
#   they validate credentials and access, but do NOT introspect schemas.
#
# - Designed for production stability:
#   explicit, minimal, defensive, and hard to misuse.


In [2]:
# ============================================================
# Cell 01 — Imports and dependency setup
# ============================================================
# Overview:
#   Establishes all shared imports and global constants required
#   across the notebook for config/state management, logging, file I/O,
#   and downstream notebook integration.
#
# Inputs / Outputs:
#   Inputs:  None (imports only)
#   Outputs: Imported modules, global constants
#
# Notes:
#   - No Notion API calls or client instantiation here
#   - Minimal external dependencies (stdlib-first approach)
#   - LLM configuration declared but not used in this notebook
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
import os
load_dotenv('env.txt')
NEWSAPI_KEY = (
    os.environ.get("NEWSAPI_API_KEY")
    or os.environ.get("NEWSAPI_KEY")
)


# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---

import sys
import json
import logging
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List
import uuid
import re

# --- Optional: config parsing ---
try:
    import yaml
    YAML_AVAILABLE = True
except ImportError:
    YAML_AVAILABLE = False
    logging.warning("PyYAML not available; config.yaml loading will be skipped")
from openai import OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

OPENAI_MODEL = "gpt-4.1-mini"
OPENAI_TEMPERATURE = 0.2
# --- Global constants ---
STATE_DIR = Path('./state')
CONFIG_FILE = Path('config.yaml')
ENV_FILE = Path('env.txt')
DEFAULT_TIMEZONE = timezone.utc

# --- Validation patterns ---
RUN_ID_PATTERN = re.compile(r'^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$')
NOTION_ID_PATTERN = re.compile(r'^[a-f0-9]{32}$')

print("✓ Cell 01: Imports and dependencies loaded")


✓ Cell 01: Imports and dependencies loaded


In [4]:
# ============================================================
# Cell 02 — Environment variable loading (researchOS v2)
# ============================================================
# Overview:
#   Load environment variables from env.txt and perform *lightweight*
#   validation for researchOS bootstrap. This cell intentionally avoids
#   strict DB enforcement; schema and connectivity checks are delegated
#   to 029_notion_clients_and_io.
#
# Inputs / Outputs:
#   Inputs:  env.txt
#   Outputs: env_config dict (normalized, downstream-safe)
#
# Notes:
#   - 028 is responsible for *bootstrap*, not strict validation
#   - NOTION DB schema validation is handled in 029
#   - Missing optional DB IDs should NOT fail Daily execution
#

from typing import Dict
import os
import re

# --- Core required env vars (minimal) ---
REQUIRED_ENV_VARS = [
    "NOTION_TOKEN",
    "NOTION_VERSION",
]

# --- Optional DB IDs used across Daily pipeline ---
OPTIONAL_DB_VARS = [
    "NOTION_LIT_DB_ID",                 # Papers
    "NOTION_EVENTS_DB_ID",              # Events
    "NOTION_MONITORING_TARGETS_DB_ID",  # Targets
    "NOTION_MONITORING_QUEUE_DB_ID",    # Queue
]

# --- Helpers ---
NOTION_ID_PATTERN = re.compile(r"^[0-9a-fA-F]{32}$")

def normalize_notion_id(value: str) -> str:
    """Normalize Notion ID by stripping hyphens."""
    return value.replace("-", "")

def mask(value: str, n: int = 6) -> str:
    return f"{value[:n]}...***" if value else "None"

# --- Load env ---
env_config: Dict[str, str] = {}
errors = []

# Required
for var in REQUIRED_ENV_VARS:
    v = os.getenv(var)
    if not v:
        errors.append(f"Missing required env var: {var}")
    else:
        env_config[var] = v

# Optional DB IDs
for var in OPTIONAL_DB_VARS:
    v = os.getenv(var)
    if not v:
        env_config[var] = ""
        continue

    nid = normalize_notion_id(v)
    if NOTION_ID_PATTERN.match(nid):
        env_config[var] = nid
    else:
        print(f"⚠ Warning: {var} looks malformed (ignored)")
        env_config[var] = ""

# --- Fail only on core secrets ---
if errors:
    raise RuntimeError(
        "Environment bootstrap failed:\n  - " + "\n  - ".join(errors)
    )
DRIVE_FOLDER_ID = os.getenv("DRIVE_FOLDER_ID")
# --- Summary ---
print("✓ Cell 02: Environment bootstrap completed")
print(f"  - NOTION_TOKEN: {mask(env_config['NOTION_TOKEN'])}")
print(f"  - NOTION_VERSION: {env_config['NOTION_VERSION']}")

for var in OPTIONAL_DB_VARS:
    v = env_config.get(var)
    status = "set" if v else "not set"
    print(f"  - {var}: {status}")

# env_config is now available for downstream cells


✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set


In [5]:
# ============================================================
# Cell 03 — Optional config.yaml loading and normalization
# ============================================================
# Overview:
#   Loads optional config.yaml file containing pipeline configuration
#   such as cadence, lookback windows, thresholds, and operational
#   parameters. Normalizes and validates configuration against schema.
#   Falls back to safe defaults if file is missing or invalid.
#
# Inputs / Outputs:
#   Inputs:  config.yaml (optional: pipeline configuration)
#   Outputs: config dict with normalized, validated settings
#
# Notes:
#   - PyYAML required for config.yaml parsing (checked in Cell 01)
#   - Missing config.yaml is NOT an error; defaults are applied
#   - Invalid config.yaml WILL raise error for safety
#   - All time windows are normalized to integers (days/hours)
#   - Configuration is validated for type safety and bounds
#

# --- Default configuration schema ---
DEFAULT_CONFIG = {
    'pipeline': {
        'cadence': 'daily',  # 'daily' | 'hourly' | 'manual'
        'max_runtime_minutes': 30,
        'enable_preflight_checks': True,
        'fail_fast': False
    },
    'lookback': {
        'daily_notes_days': 7,
        'tasks_days': 14,
        'projects_days': 30
    },
    'thresholds': {
        'max_items_per_query': 100,
        'max_concurrent_updates': 10,
        'rate_limit_delay_ms': 333  # ~3 req/sec
    },
    'processing': {
        'skip_archived': True,
        'skip_deleted': True,
        'validate_schemas': True,
        'deduplicate_results': True
    },
    'state': {
        'save_intermediate': True,
        'backup_on_error': True,
        'max_state_files': 10
    },
    'logging': {
        'level': 'INFO',  # DEBUG | INFO | WARNING | ERROR
        'log_to_file': True,
        'log_to_console': True,
        'verbose_api_calls': False
    }
}

# --- Configuration schema validation rules ---
CONFIG_SCHEMA = {
    'pipeline.cadence': {'type': str, 'allowed': ['daily', 'hourly', 'manual']},
    'pipeline.max_runtime_minutes': {'type': int, 'min': 1, 'max': 120},
    'lookback.daily_notes_days': {'type': int, 'min': 1, 'max': 90},
    'lookback.tasks_days': {'type': int, 'min': 1, 'max': 180},
    'lookback.projects_days': {'type': int, 'min': 1, 'max': 365},
    'thresholds.max_items_per_query': {'type': int, 'min': 1, 'max': 100},
    'thresholds.max_concurrent_updates': {'type': int, 'min': 1, 'max': 50},
    'logging.level': {'type': str, 'allowed': ['DEBUG', 'INFO', 'WARNING', 'ERROR']}
}

def deep_merge(base: Dict, overlay: Dict) -> Dict:
    """
    Deep merge overlay dict into base dict, preserving nested structure.
    Overlay values take precedence over base values.
    """
    result = base.copy()
    
    for key, value in overlay.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    
    return result

def get_nested(d: Dict, path: str, default=None):
    """
    Get nested dict value using dot-notation path.
    Example: get_nested(config, 'pipeline.cadence')
    """
    keys = path.split('.')
    current = d
    
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    
    return current

def validate_config_value(path: str, value: Any, schema: Dict) -> None:
    """
    Validate a single config value against schema rules.
    Raises ValueError if validation fails.
    """
    if path not in schema:
        return  # No validation rule defined
    
    rule = schema[path]
    expected_type = rule['type']
    
    # Type check
    if not isinstance(value, expected_type):
        raise ValueError(
            f"Config '{path}': expected {expected_type.__name__}, "
            f"got {type(value).__name__} ({value!r})"
        )
    
    # Allowed values check
    if 'allowed' in rule:
        if value not in rule['allowed']:
            raise ValueError(
                f"Config '{path}': value '{value}' not in allowed set {rule['allowed']}"
            )
    
    # Range checks for numeric values
    if 'min' in rule and value < rule['min']:
        raise ValueError(
            f"Config '{path}': value {value} below minimum {rule['min']}"
        )
    
    if 'max' in rule and value > rule['max']:
        raise ValueError(
            f"Config '{path}': value {value} above maximum {rule['max']}"
        )

def validate_config(config: Dict, schema: Dict) -> List[str]:
    """
    Validate entire config dict against schema.
    Returns list of validation errors (empty if valid).
    """
    errors = []
    
    for path, rule in schema.items():
        try:
            value = get_nested(config, path)
            if value is not None:
                validate_config_value(path, value, schema)
        except ValueError as e:
            errors.append(str(e))
    
    return errors

# --- Load config.yaml if available ---
config = DEFAULT_CONFIG.copy()
config_source = 'defaults'

if CONFIG_FILE.exists():
    if not YAML_AVAILABLE:
        print(f"⚠ Cell 03: {CONFIG_FILE} found but PyYAML not available")
        print("  Using default configuration")
    else:
        try:
            with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
                user_config = yaml.safe_load(f)
            
            if user_config is None:
                user_config = {}
            
            if not isinstance(user_config, dict):
                raise ValueError(
                    f"Config root must be dict, got {type(user_config).__name__}"
                )
            
            # Merge user config over defaults
            config = deep_merge(DEFAULT_CONFIG, user_config)
            config_source = str(CONFIG_FILE)
            
            print(f"✓ Cell 03: Loaded configuration from {CONFIG_FILE}")
        
        except yaml.YAMLError as e:
            raise RuntimeError(f"Failed to parse {CONFIG_FILE}: {e}")
        except Exception as e:
            raise RuntimeError(f"Failed to load {CONFIG_FILE}: {e}")
else:
    print(f"ℹ Cell 03: {CONFIG_FILE} not found, using defaults")

# --- Validate loaded/merged configuration ---
validation_errors = validate_config(config, CONFIG_SCHEMA)

if validation_errors:
    error_report = "\n  - ".join(
        ['Configuration validation failed:'] + validation_errors
    )
    raise ValueError(error_report)

# --------------------------------------------------
# Drive configuration (normalized, optional)
#   Priority: env (DRIVE_FOLDER_ID) > config.yaml > None
# --------------------------------------------------
config.setdefault("drive", {})
env_drive_folder_id = os.getenv("DRIVE_FOLDER_ID")

if env_drive_folder_id:
    config["drive"]["folder_id"] = env_drive_folder_id
else:
    # keep whatever config.yaml provided (or None)
    config["drive"]["folder_id"] = config["drive"].get("folder_id")

if config.get("drive", {}).get("folder_id"):
    print("  - Drive folder_id:      (set)")
else:
    print("  - Drive folder_id:      (not set)")

# --- Success summary ---
print(f"✓ Cell 03: Configuration validated (source: {config_source})")
print(f"  - Pipeline cadence:     {config['pipeline']['cadence']}")
print(f"  - Max runtime:          {config['pipeline']['max_runtime_minutes']} min")
print(f"  - Lookback (daily):     {config['lookback']['daily_notes_days']} days")
print(f"  - Lookback (tasks):     {config['lookback']['tasks_days']} days")
print(f"  - Lookback (projects):  {config['lookback']['projects_days']} days")
print(f"  - Max items per query:  {config['thresholds']['max_items_per_query']}")
print(f"  - Logging level:        {config['logging']['level']}")

# --- Export for downstream use ---
# config dict is now available for other cells/notebooks


✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO


In [6]:
# ============================================================
# Cell 04 — Run context initialization (run_id, timestamps)
# ============================================================
# Overview:
#   Generates a unique run_id for this pipeline execution and captures
#   execution timestamps. The run_id provides traceability across logs,
#   state files, and downstream notebook calls. Timestamps establish
#   temporal bounds for lookback windows and state versioning.
#
# Inputs / Outputs:
#   Inputs:  None (uses system time and UUID generation)
#   Outputs: run_context dict with run_id, timestamps, and metadata
#
# Notes:
#   - run_id follows UUID4 format (lowercase hex with hyphens)
#   - All timestamps use UTC timezone for consistency
#   - execution_start marks pipeline begin time
#   - run_date represents the logical "as-of" date for the run
#   - Run context is exported for use in logging, state files, and notebooks
#

# --- Generate unique run identifier ---
run_id = str(uuid.uuid4())

# Validate run_id format (paranoid check)
if not RUN_ID_PATTERN.match(run_id):
    raise RuntimeError(
        f"Generated run_id '{run_id}' does not match expected UUID4 format"
    )

# --- Capture execution timestamps ---
execution_start = datetime.now(DEFAULT_TIMEZONE)

# Logical run date (normalized to date boundary)
# This represents the "as-of" date for the pipeline run
run_date = execution_start.date()

# ISO 8601 formatted strings for serialization
execution_start_iso = execution_start.isoformat()
run_date_iso = run_date.isoformat()

# --- Build run context dictionary ---
run_context = {
    'run_id': run_id,
    'execution_start': execution_start,
    'execution_start_iso': execution_start_iso,
    'run_date': run_date,
    'run_date_iso': run_date_iso,
    'timezone': str(DEFAULT_TIMEZONE),
    'config_source': config_source,
    'cadence': config['pipeline']['cadence'],
    'notebook': '028_config_and_state',
    'notebook_version': '1.0.0'
}

# --- Calculate lookback windows based on config ---
from datetime import timedelta

lookback_windows = {
    'daily_notes': {
        'days': config['lookback']['daily_notes_days'],
        'start_date': run_date - timedelta(days=config['lookback']['daily_notes_days']),
        'end_date': run_date
    },
    'tasks': {
        'days': config['lookback']['tasks_days'],
        'start_date': run_date - timedelta(days=config['lookback']['tasks_days']),
        'end_date': run_date
    },
    'projects': {
        'days': config['lookback']['projects_days'],
        'start_date': run_date - timedelta(days=config['lookback']['projects_days']),
        'end_date': run_date
    }
}

# Add lookback windows to run context
run_context['lookback_windows'] = {
    db: {
        'days': window['days'],
        'start_date_iso': window['start_date'].isoformat(),
        'end_date_iso': window['end_date'].isoformat()
    }
    for db, window in lookback_windows.items()
}

# --- Success summary ---
print("✓ Cell 04: Run context initialized")
print(f"  - Run ID:           {run_id}")
print(f"  - Execution start:  {execution_start_iso}")
print(f"  - Run date:         {run_date_iso}")
print(f"  - Timezone:         {DEFAULT_TIMEZONE}")
print(f"  - Cadence:          {config['pipeline']['cadence']}")
print(f"  - Lookback windows:")
for db, window in lookback_windows.items():
    print(f"    - {db:12s}: {window['days']} days ({window['start_date']} to {window['end_date']})")

# --- Export for downstream use ---
# run_context dict is now available for other cells/notebooks
# lookback_windows dict provides temporal bounds for queries


✓ Cell 04: Run context initialized
  - Run ID:           775d9a47-7a9a-4ddf-bf9c-e82c22063cd9
  - Execution start:  2026-01-28T19:25:58.554602+00:00
  - Run date:         2026-01-28
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-21 to 2026-01-28)
    - tasks       : 14 days (2026-01-14 to 2026-01-28)
    - projects    : 30 days (2025-12-29 to 2026-01-28)


In [7]:
# ============================================================
# Cell 05 — State directory setup and verification
# ============================================================
# Overview:
#   Creates and verifies the state directory structure for persisting
#   pipeline execution state, cursors, processed item IDs, and run
#   metadata. Ensures directory permissions and validates existing
#   state files for integrity.
#
# Inputs / Outputs:
#   Inputs:  STATE_DIR constant (from Cell 01), run_context (from Cell 04)
#   Outputs: Verified state directory, state_paths dict with file locations
#
# Notes:
#   - Creates ./state/ directory if missing
#   - Validates write permissions before proceeding
#   - Checks existing state files for JSON validity
#   - Does NOT load state content (deferred to Cell 06)
#   - Establishes naming convention for state files
#   - Implements optional state file rotation based on config
#

# --- State directory structure ---
STATE_SUBDIRS = [
    'cursors',      # Notion sync cursors for incremental updates
    'processed',    # Sets of processed item IDs for deduplication
    'runs',         # Per-run metadata and execution logs
    'backups'       # Automatic backups on error (if enabled)
]

# --- State file naming conventions ---
def get_state_file_path(category: str, name: str, run_id: Optional[str] = None) -> Path:
    """
    Generate standardized state file path.
    
    Args:
        category: Subdirectory (cursors | processed | runs | backups)
        name: Base filename without extension
        run_id: Optional run identifier for run-specific files
    
    Returns:
        Path object for state file
    
    Examples:
        get_state_file_path('cursors', 'daily_notes') -> ./state/cursors/daily_notes.json
        get_state_file_path('runs', 'metadata', run_id) -> ./state/runs/{run_id}_metadata.json
    """
    if category not in STATE_SUBDIRS:
        raise ValueError(f"Invalid state category '{category}', expected one of {STATE_SUBDIRS}")
    
    subdir = STATE_DIR / category
    
    if run_id:
        filename = f"{run_id}_{name}.json"
    else:
        filename = f"{name}.json"
    
    return subdir / filename

# --- Create state directory structure ---
print(f"Setting up state directory: {STATE_DIR.absolute()}")

try:
    # Create root state directory
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    
    # Create subdirectories
    for subdir_name in STATE_SUBDIRS:
        subdir = STATE_DIR / subdir_name
        subdir.mkdir(exist_ok=True)
        print(f"  ✓ Created/verified: {subdir.relative_to(STATE_DIR.parent)}")
    
except PermissionError as e:
    raise RuntimeError(
        f"Permission denied creating state directory {STATE_DIR}: {e}"
    )
except Exception as e:
    raise RuntimeError(
        f"Failed to create state directory structure: {e}"
    )

# --- Verify write permissions ---
test_file = STATE_DIR / '.write_test'
try:
    test_file.write_text('test', encoding='utf-8')
    test_file.unlink()
    print(f"  ✓ Write permissions verified")
except Exception as e:
    raise RuntimeError(
        f"State directory {STATE_DIR} is not writable: {e}"
    )

# --- Define standard state file paths ---
state_paths = {
    # Sync cursors for incremental updates
    'cursor_daily': get_state_file_path('cursors', 'daily_notes'),
    'cursor_tasks': get_state_file_path('cursors', 'tasks'),
    'cursor_projects': get_state_file_path('cursors', 'projects'),
    
    # Processed item ID sets for deduplication
    'processed_daily': get_state_file_path('processed', 'daily_notes_ids'),
    'processed_tasks': get_state_file_path('processed', 'task_ids'),
    'processed_projects': get_state_file_path('processed', 'project_ids'),
    
    # Run-specific metadata
    'run_metadata': get_state_file_path('runs', 'metadata', run_id),
    'run_summary': get_state_file_path('runs', 'summary', run_id),
    
    # Latest run pointer (symlink or copy)
    'latest_run': STATE_DIR / 'runs' / 'latest.json'
}

# --- Validate existing state files ---
state_file_status = {}

for key, path in state_paths.items():
    if path.exists():
        try:
            # Verify file is readable and valid JSON
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
                if content.strip():  # Non-empty file
                    json.loads(content)  # Validate JSON structure
            
            state_file_status[key] = {
                'exists': True,
                'valid': True,
                'size_bytes': path.stat().st_size,
                'modified': datetime.fromtimestamp(
                    path.stat().st_mtime, 
                    tz=DEFAULT_TIMEZONE
                ).isoformat()
            }
        
        except json.JSONDecodeError as e:
            state_file_status[key] = {
                'exists': True,
                'valid': False,
                'error': f"Invalid JSON: {e}"
            }
            print(f"  ⚠ Corrupted state file: {path.name} ({e})")
        
        except Exception as e:
            state_file_status[key] = {
                'exists': True,
                'valid': False,
                'error': str(e)
            }
            print(f"  ⚠ Cannot read state file: {path.name} ({e})")
    else:
        state_file_status[key] = {
            'exists': False,
            'valid': None
        }

# --- Implement state file rotation (if configured) ---
if config['state']['max_state_files'] > 0:
    runs_dir = STATE_DIR / 'runs'
    
    # Get all run metadata files (excluding latest.json)
    run_files = sorted(
        [f for f in runs_dir.glob('*_metadata.json') if f.name != 'latest.json'],
        key=lambda p: p.stat().st_mtime,
        reverse=True  # Newest first
    )
    
    max_files = config['state']['max_state_files']
    
    if len(run_files) > max_files:
        files_to_remove = run_files[max_files:]
        print(f"  Rotating old state files (keeping {max_files} most recent):")
        
        for old_file in files_to_remove:
            try:
                # Also remove corresponding summary file
                summary_file = old_file.parent / old_file.name.replace('_metadata', '_summary')
                
                old_file.unlink()
                if summary_file.exists():
                    summary_file.unlink()
                
                print(f"    - Removed: {old_file.name}")
            
            except Exception as e:
                print(f"    ⚠ Failed to remove {old_file.name}: {e}")

# --- Summary ---
print("\n✓ Cell 05: State directory setup complete")
print(f"  - Root directory:     {STATE_DIR.absolute()}")
print(f"  - Subdirectories:     {len(STATE_SUBDIRS)}")
print(f"  - Tracked state files: {len(state_paths)}")

existing_valid = sum(1 for s in state_file_status.values() if s['exists'] and s.get('valid'))
existing_invalid = sum(1 for s in state_file_status.values() if s['exists'] and not s.get('valid'))
new_files = sum(1 for s in state_file_status.values() if not s['exists'])

print(f"  - Existing valid:     {existing_valid}")
print(f"  - Existing invalid:   {existing_invalid}")
print(f"  - New (will create):  {new_files}")

if existing_invalid > 0:
    print(f"  ⚠ Warning: {existing_invalid} corrupted state file(s) detected")
    print(f"    These will be backed up and recreated on first write")

# --- Export for downstream use ---
# state_paths dict: standardized paths for all state files
# state_file_status dict: validation results for existing files
# get_state_file_path(): helper function for generating state file paths


Setting up state directory: /Users/yuetoya/Desktop/researchOS100-private/notebooks/state
  ✓ Created/verified: state/cursors
  ✓ Created/verified: state/processed
  ✓ Created/verified: state/runs
  ✓ Created/verified: state/backups
  ✓ Write permissions verified
  Rotating old state files (keeping 10 most recent):
    - Removed: bcea8a95-6abd-4d3f-9737-6ae448be0663_metadata.json

✓ Cell 05: State directory setup complete
  - Root directory:     /Users/yuetoya/Desktop/researchOS100-private/notebooks/state
  - Subdirectories:     4
  - Tracked state files: 9
  - Existing valid:     0
  - Existing invalid:   0
  - New (will create):  9


In [8]:
# ============================================================
# Cell 06 — State persistence helpers (JSON read/write)
# ============================================================
# Overview:
#   Robust JSON-based state persistence with atomic writes, optional backups,
#   and light validation. Used for cursors, processed ID sets, run metadata,
#   and other pipeline state.
#
# Inputs / Outputs:
#   Inputs:  state_paths dict (from Cell 05), config (from Cell 03)
#   Outputs: load_state(), save_state(), backup_state_file(), bulk helpers
#
# Notes:
#   - Atomic writes (temp + rename)
#   - Optional backup on overwrite
#   - Missing/empty files return defaults (not errors)
#   - JSON size warning only (does not fail)
#

from typing import Set, Union
import shutil

StateValue = Union[Dict[str, Any], List[Any], Set[str], str, int, float, bool, None]

# --- Atomic write implementation ---
def atomic_write_json(path: Path, data: Any, indent: int = 2) -> None:
    temp_path = path.parent / f".{path.name}.tmp.{uuid.uuid4().hex[:8]}"
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        temp_path.replace(path)

    except Exception as e:
        if temp_path.exists():
            try:
                temp_path.unlink()
            except Exception:
                pass
        raise RuntimeError(f"Failed to write {path}: {e}")

# --- Backup creation ---
def backup_state_file(path: Path) -> Optional[Path]:
    if not path.exists():
        return None

    backups_dir = STATE_DIR / "backups"
    backups_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(DEFAULT_TIMEZONE).strftime("%Y%m%d_%H%M%S")
    backup_name = f"{path.stem}_{timestamp}.backup.json"
    backup_path = backups_dir / backup_name

    try:
        shutil.copy2(path, backup_path)
        return backup_path
    except Exception as e:
        raise RuntimeError(f"Failed to backup {path} -> {backup_path}: {e}")

# --- State loading ---
def load_state(key: str, default: StateValue = None, validate: bool = True) -> StateValue:
    if key not in state_paths:
        raise ValueError(f"Unknown state key '{key}', expected one of {list(state_paths.keys())}")

    path = state_paths[key]
    if not path.exists():
        return default

    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        if not content:
            return default

        data = json.loads(content)

        if validate:
            size_mb = path.stat().st_size / (1024 * 1024)
            if size_mb > 10:
                print(f"  ⚠ Warning: Large state file {path.name} ({size_mb:.1f} MB)")

        return data

    except json.JSONDecodeError as e:
        print(f"  ⚠ Warning: Invalid JSON in {path.name}: {e} -> returning default")
        return default
    except Exception as e:
        print(f"  ⚠ Warning: Failed to load {path.name}: {e} -> returning default")
        return default

# --- State saving ---
def save_state(
    key: str,
    data: StateValue,
    backup: Optional[bool] = None,
    metadata: Optional[Dict[str, Any]] = None,
) -> bool:
    if key not in state_paths:
        raise ValueError(f"Unknown state key '{key}', expected one of {list(state_paths.keys())}")

    path = state_paths[key]

    # Decide backup behavior (prefer explicit config key; fall back safely)
    if backup is None:
        backup = bool(config.get("state", {}).get("backup_on_overwrite", False))

    try:
        # Convert sets to sorted lists for JSON serialization
        if isinstance(data, set):
            data = sorted(list(data))

        # Merge metadata only if dict
        if metadata:
            if isinstance(data, dict):
                data = {**data, **metadata}
            else:
                print(f"  ⚠ Warning: metadata ignored for non-dict state '{key}'")

        # Backup existing file before overwrite (if enabled)
        if backup and path.exists():
            bp = backup_state_file(path)
            if bp:
                print(f"  Created backup: {bp.name}")

        atomic_write_json(path, data)
        return True

    except Exception as e:
        print(f"  ✗ Failed to save state to {path.name}: {e}")
        return False

# --- Bulk ops ---
def save_run_state(state_dict: Dict[str, StateValue]) -> Dict[str, bool]:
    results = {}
    for key, data in state_dict.items():
        try:
            results[key] = save_state(key, data)
        except Exception as e:
            print(f"  ✗ Error saving {key}: {e}")
            results[key] = False
    return results

def load_run_state(keys: List[str]) -> Dict[str, StateValue]:
    results = {}
    for key in keys:
        try:
            results[key] = load_state(key)
        except Exception as e:
            print(f"  ✗ Error loading {key}: {e}")
            results[key] = None
    return results

# --- Integrity check ---
def validate_state_integrity() -> Dict[str, Any]:
    results = {"valid": [], "invalid": [], "missing": [], "warnings": []}
    for key, path in state_paths.items():
        if not path.exists():
            results["missing"].append(key)
            continue
        try:
            data = load_state(key, validate=True)
            if data is None:
                results["missing"].append(key)
            else:
                results["valid"].append(key)
        except Exception as e:
            results["invalid"].append({"key": key, "error": str(e)})
    return results

# --- Initialize run metadata state ---
initial_run_metadata = {
    "run_id": run_id,
    "execution_start_iso": execution_start_iso,
    "run_date_iso": run_date_iso,
    "config_source": config_source,
    "cadence": config["pipeline"]["cadence"],
    "lookback_windows": run_context["lookback_windows"],
    "state_version": "1.0.0",
    "status": "initialized",
}

save_state("run_metadata", initial_run_metadata, backup=False)

print("✓ Cell 06: State persistence helpers loaded")
print("  - Functions: load_state(), save_state(), backup_state_file()")
print("  - Bulk ops:  save_run_state(), load_run_state()")
print("  - Validation: validate_state_integrity()")
print(f"  - Initial run metadata saved: {state_paths['run_metadata'].name}")

integrity_check = validate_state_integrity()
print(f"  - Valid state files:   {len(integrity_check['valid'])}")
print(f"  - Missing state files: {len(integrity_check['missing'])}")
print(f"  - Invalid state files: {len(integrity_check['invalid'])}")

if integrity_check["invalid"]:
    print("  ⚠ Invalid state files detected:")
    for item in integrity_check["invalid"]:
        print(f"    - {item['key']}: {item['error']}")


✓ Cell 06: State persistence helpers loaded
  - Functions: load_state(), save_state(), backup_state_file()
  - Bulk ops:  save_run_state(), load_run_state()
  - Validation: validate_state_integrity()
  - Initial run metadata saved: 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9_metadata.json
  - Valid state files:   1
  - Missing state files: 8
  - Invalid state files: 0


In [9]:
# ============================================================
# Cell 07 — Logging configuration with run_id injection
# ============================================================
# Overview:
#   Configures structured logging for the pipeline with run_id context
#   injection, file and console handlers, and appropriate formatting.
#   Enables correlation of log entries across cells and notebooks within
#   a single pipeline execution.
#
# Inputs / Outputs:
#   Inputs:  config (from Cell 03), run_context (from Cell 04), STATE_DIR (from Cell 01)
#   Outputs: Configured logger instance, log file path, get_logger() helper
#
# Notes:
#   - run_id automatically injected into all log records
#   - Log level controlled via config['logging']['level']
#   - File logging writes to ./state/runs/{run_id}.log
#   - Console logging includes colored output (if supported)
#   - Downstream notebooks can use get_logger(__name__) pattern
#   - Log rotation not implemented (managed by external tooling)
#

# --- Log directory setup ---
LOG_DIR = STATE_DIR / 'runs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Log file for this run
log_file_path = LOG_DIR / f"{run_id}.log"

# --- Custom log formatter with run_id ---
class RunContextFormatter(logging.Formatter):
    """
    Custom formatter that injects run_id into every log record.
    """
    
    def __init__(self, fmt: str, datefmt: str = None, run_id: str = None):
        super().__init__(fmt, datefmt)
        self.run_id = run_id or 'unknown'
    
    def format(self, record: logging.LogRecord) -> str:
        # Inject run_id into record if not already present
        if not hasattr(record, 'run_id'):
            record.run_id = self.run_id
        
        return super().format(record)

# --- Log format strings ---
FILE_LOG_FORMAT = (
    '%(asctime)s | %(levelname)-8s | %(run_id)s | '
    '%(name)s:%(funcName)s:%(lineno)d | %(message)s'
)

CONSOLE_LOG_FORMAT = (
    '%(asctime)s | %(levelname)-8s | %(run_id)s | %(message)s'
)

DATE_FORMAT = '%Y-%m-%d %H:%M:%S'

# --- Parse log level from config ---
log_level_str = config['logging']['level'].upper()
log_level_map = {
    'DEBUG': logging.DEBUG,
    'INFO': logging.INFO,
    'WARNING': logging.WARNING,
    'ERROR': logging.ERROR,
    'CRITICAL': logging.CRITICAL
}

log_level = log_level_map.get(log_level_str, logging.INFO)

# --- Configure root logger ---
root_logger = logging.getLogger()
root_logger.setLevel(log_level)

# Remove any existing handlers (avoid duplicates in notebook re-runs)
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)

# --- File handler (always enabled for audit trail) ---
file_handler = logging.FileHandler(log_file_path, mode='a', encoding='utf-8')
file_handler.setLevel(log_level)
file_formatter = RunContextFormatter(FILE_LOG_FORMAT, DATE_FORMAT, run_id)
file_handler.setFormatter(file_formatter)
root_logger.addHandler(file_handler)

print(f"✓ File logging enabled: {log_file_path.relative_to(STATE_DIR.parent)}")

# --- Console handler (conditional based on config) ---
if config['logging']['log_to_console']:
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(log_level)
    console_formatter = RunContextFormatter(CONSOLE_LOG_FORMAT, DATE_FORMAT, run_id)
    console_handler.setFormatter(console_formatter)
    root_logger.addHandler(console_handler)
    
    print(f"✓ Console logging enabled")
else:
    print(f"ℹ Console logging disabled (per config)")

# --- Helper function for downstream notebooks ---
def get_logger(name: str) -> logging.Logger:
    """
    Get a logger instance for a module/notebook with run_id context.
    
    Args:
        name: Logger name (typically __name__ or notebook identifier)
    
    Returns:
        Configured logger instance
    
    Example:
        logger = get_logger('036_daily_orchestrator')
        logger.info('Starting daily pipeline')
    """
    logger = logging.getLogger(name)
    
    # Ensure run_id is available in extra context
    # (handlers will inject it via RunContextFormatter)
    return logger

# --- Create module-level logger for this notebook ---
logger = get_logger('028_config_and_state')

# --- Test logging configuration ---
logger.info("Logging configuration initialized")
logger.debug(f"Log level: {log_level_str}")
logger.debug(f"File logging: {log_file_path}")
logger.debug(f"Console logging: {config['logging']['log_to_console']}")

# --- Log run context initialization ---
logger.info(f"Pipeline run started: {run_id}")
logger.info(f"Execution time: {execution_start_iso}")
logger.info(f"Run date: {run_date_iso}")
logger.info(f"Cadence: {config['pipeline']['cadence']}")
logger.info(f"Config source: {config_source}")

# --- Verbose API logging setup (if enabled) ---
if config['logging'].get('verbose_api_calls', False):
    # Set higher log level for HTTP libraries to see API calls
    logging.getLogger('urllib3').setLevel(logging.DEBUG)
    logging.getLogger('requests').setLevel(logging.DEBUG)
    logger.info("Verbose API call logging enabled")
else:
    # Suppress noisy HTTP library logs
    logging.getLogger('urllib3').setLevel(logging.WARNING)
    logging.getLogger('requests').setLevel(logging.WARNING)

# --- Success summary ---
print("\n✓ Cell 07: Logging configuration complete")
print(f"  - Log level:        {log_level_str}")
print(f"  - Log file:         {log_file_path.name}")
print(f"  - Console output:   {config['logging']['log_to_console']}")
print(f"  - Verbose API logs: {config['logging'].get('verbose_api_calls', False)}")
print(f"  - Run ID injected:  {run_id}")

# --- Export for downstream use ---
# logger: Module-level logger for this notebook
# get_logger(name): Factory function for creating loggers in other notebooks
# log_file_path: Path to current run's log file
# RunContextFormatter: Custom formatter class (if needed for extension)


✓ File logging enabled: state/runs/775d9a47-7a9a-4ddf-bf9c-e82c22063cd9.log
✓ Console logging enabled
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Logging configuration initialized
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Pipeline run started: 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Execution time: 2026-01-28T19:25:58.554602+00:00
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Run date: 2026-01-28
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Cadence: daily
2026-01-29 04:26:01 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Config source: config.yaml

✓ Cell 07: Logging configuration complete
  - Log level:        INFO
  - Log file:         775d9a47-7a9a-4ddf-bf9c-e82c22063cd9.log
  - Console output:   True
  - Verbose API logs: False
  - Run ID injected:  775d9a47-7a9a-4ddf-bf9c-e82c22063cd9


In [10]:
# ============================================================
# Cell 08 — Lightweight Notion preflight checks (029 optional, token-aware)
# ============================================================
from datetime import datetime
from typing import Optional, Tuple, Dict, Any
import os
import json
import requests

enable_checks = bool(config.get("pipeline", {}).get("enable_preflight_checks", True))
fail_fast = bool(config.get("pipeline", {}).get("fail_fast", True))

def _mask(s: str, n: int = 6) -> str:
    if not s:
        return ""
    return s[:n] + "...***"

def _pick_token() -> Tuple[Optional[str], str]:
    """
    Pick token from (globals/env_config/os.environ) with name drift support.
    Priority:
      1) NOTION_TOKEN
      2) NOTION_API_KEY   (legacy)
    Returns: (token, source_label)
    """
    # globals
    if "NOTION_TOKEN" in globals() and globals().get("NOTION_TOKEN"):
        return globals()["NOTION_TOKEN"], "globals.NOTION_TOKEN"
    if "NOTION_API_KEY" in globals() and globals().get("NOTION_API_KEY"):
        return globals()["NOTION_API_KEY"], "globals.NOTION_API_KEY"

    # env_config
    if "env_config" in globals() and isinstance(env_config, dict):
        if env_config.get("NOTION_TOKEN"):
            return env_config["NOTION_TOKEN"], "env_config.NOTION_TOKEN"
        if env_config.get("NOTION_API_KEY"):
            return env_config["NOTION_API_KEY"], "env_config.NOTION_API_KEY"

    # OS env
    if os.getenv("NOTION_TOKEN"):
        return os.getenv("NOTION_TOKEN"), "os.environ.NOTION_TOKEN"
    if os.getenv("NOTION_API_KEY"):
        return os.getenv("NOTION_API_KEY"), "os.environ.NOTION_API_KEY"

    return None, "missing"

def _pick_version() -> Tuple[str, str]:
    """
    Pick Notion-Version similarly.
    """
    if "NOTION_VERSION" in globals() and globals().get("NOTION_VERSION"):
        return globals()["NOTION_VERSION"], "globals.NOTION_VERSION"
    if "env_config" in globals() and isinstance(env_config, dict) and env_config.get("NOTION_VERSION"):
        return env_config["NOTION_VERSION"], "env_config.NOTION_VERSION"
    if os.getenv("NOTION_VERSION"):
        return os.getenv("NOTION_VERSION"), "os.environ.NOTION_VERSION"
    # fallback
    return "2022-06-28", "default(2022-06-28)"

def _base_url() -> str:
    if "NOTION_API_BASE_URL" in globals() and globals().get("NOTION_API_BASE_URL"):
        return str(globals()["NOTION_API_BASE_URL"]).rstrip("/")
    if os.getenv("NOTION_API_BASE_URL"):
        return os.getenv("NOTION_API_BASE_URL").rstrip("/")
    return "https://api.notion.com/v1"

def _ensure_session(token: str, version: str) -> requests.Session:
    """
    Ensure we have a requests.Session with correct headers.
    IMPORTANT: overwrite Authorization/Notion-Version to avoid stale headers.
    """
    sess = globals().get("session") if "session" in globals() else None
    if not isinstance(sess, requests.Session):
        sess = requests.Session()

    sess.headers.update({
        "Authorization": f"Bearer {token}",
        "Notion-Version": version,
        "Content-Type": "application/json",
    })
    return sess

def _rest_get(sess: requests.Session, base: str, path: str, timeout: int = 10) -> requests.Response:
    return sess.get(f"{base}{path}", timeout=timeout)

def _get_db_id(name: str) -> Optional[str]:
    """
    Resolve DB IDs with alias support.

    Priority:
      1) globals() common names
      2) env_config common keys
      3) config['databases'] keys
    """

    # 1) globals aliases (most reliable inside notebooks)
    global_aliases = {
        "papers": [
            "PAPERS_DB_ID",
            "LIT_DB_ID",
            "NOTION_LIT_DB_ID",
            "NOTION_PAPERS_DB_ID",
        ],
        "events": [
            "EVENTS_DB_ID",
            "NOTION_EVENTS_DB_ID",
        ],
        "monitoring_targets": [
            "MONITORING_TARGETS_DB_ID",
            "NOTION_MONITORING_TARGETS_DB_ID",
        ],
        "monitoring_queue": [
            "MONITORING_QUEUE_DB_ID",
            "NOTION_MONITORING_QUEUE_DB_ID",
        ],
    }

    for key in global_aliases.get(name, []):
        if key in globals() and globals().get(key):
            return globals()[key]

    # 2) env_config aliases
    if "env_config" in globals() and isinstance(env_config, dict):
        env_aliases = {
            "papers": [
                "PAPERS_DB_ID",
                "LIT_DB_ID",
                "NOTION_LIT_DB_ID",
                "NOTION_PAPERS_DB_ID",
            ],
            "events": [
                "EVENTS_DB_ID",
                "NOTION_EVENTS_DB_ID",
            ],
            "monitoring_targets": [
                "MONITORING_TARGETS_DB_ID",
                "NOTION_MONITORING_TARGETS_DB_ID",
            ],
            "monitoring_queue": [
                "MONITORING_QUEUE_DB_ID",
                "NOTION_MONITORING_QUEUE_DB_ID",
            ],
        }
        for key in env_aliases.get(name, []):
            if env_config.get(key):
                return env_config[key]

        # also support pattern keys if you ever switch naming
        # e.g. NOTION_PAPERS_DB_ID, NOTION_EVENTS_DB_ID, etc.
        pattern_key = f"NOTION_{name.upper()}_DB_ID"
        if env_config.get(pattern_key):
            return env_config[pattern_key]

    # 3) config.yaml/databases aliases
    db_cfg = (config.get("databases", {}) if isinstance(config, dict) else {})
    cfg_aliases = {
        "papers": ["papers_db_id", "lit_db_id", "literature_db_id"],
        "events": ["events_db_id"],
        "monitoring_targets": ["monitoring_targets_db_id"],
        "monitoring_queue": ["monitoring_queue_db_id"],
    }
    for k in cfg_aliases.get(name, []):
        if db_cfg.get(k):
            return db_cfg[k]

    return None

# ---------- MAIN ----------
if not enable_checks:
    logger.info("Preflight checks disabled via configuration")
    print("ℹ Cell 08: Preflight checks disabled (per config)")
    preflight_results = {"enabled": False, "skipped": True, "checks": {}, "overall_status": "skipped"}

else:
    logger.info("Starting Notion preflight checks (029 optional)")
    print("Running Notion preflight checks (lightweight)...")

    preflight_results: Dict[str, Any] = {"enabled": True, "skipped": False, "checks": {}, "overall_status": "pending"}

    token, token_src = _pick_token()
    version, version_src = _pick_version()
    base = _base_url()

    # 0) Token presence check
    if not token:
        msg = (
            "Notion token is missing.\n"
            "Set NOTION_TOKEN (recommended) or NOTION_API_KEY (legacy) in env.txt / environment."
        )
        preflight_results["checks"]["authentication"] = {
            "authenticated": False,
            "error": msg,
            "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
        }
        print("  ✗ Authentication: FAILED (token missing)")
        logger.error(msg)
        preflight_results["overall_status"] = "failed"
        if fail_fast:
            raise RuntimeError(msg)
    else:
        # Ensure session has correct headers (avoid stale/old token header)
        session = _ensure_session(token=token, version=version)

        # 1) Auth
        auth_ok = False
        auth_err: Optional[str] = None
        try:
            resp = _rest_get(session, base, "/users/me", timeout=10)
            if resp.status_code == 200:
                auth_ok = True
            else:
                # keep short but informative
                auth_err = f"HTTP {resp.status_code}: {resp.text}"
        except Exception as e:
            auth_err = str(e)

        preflight_results["checks"]["authentication"] = {
            "authenticated": auth_ok,
            "token_source": token_src,
            "token_prefix": _mask(token, 6),
            "token_length": len(token),
            "notion_version": version,
            "notion_version_source": version_src,
            "base_url": base,
            "error": auth_err,
            "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
        }

        if auth_ok:
            print("  ✓ Authentication: VERIFIED")
        else:
            # IMPORTANT: if 401, stop DB checks to reduce noise
            print(f"  ✗ Authentication: FAILED ({auth_err})")
            logger.error("Authentication failed: %s", auth_err)

            hint = (
                "Action:\n"
                "  - Confirm env.txt has the correct integration token.\n"
                "  - Use NOTION_TOKEN (recommended). If you used NOTION_API_KEY, rename/mirror it.\n"
                "  - Make sure the token belongs to an Internal Integration and is NOT expired.\n"
                "  - Share the target Notion pages/databases with that integration.\n"
                f"  - Token loaded from: {token_src} (prefix={_mask(token,6)}, len={len(token)})\n"
            )
            preflight_results["overall_status"] = "failed"
            preflight_results["checks"]["authentication"]["hint"] = hint
            print(hint.rstrip())

            if fail_fast:
                raise RuntimeError("Notion authentication failed. Fix token and rerun.")
            else:
                print("  ⚠ Skipping DB checks because authentication failed (fail_fast=False).")

        # 2) DB checks only if auth_ok
        if auth_ok:
            targets = ["papers", "events", "monitoring_targets", "monitoring_queue"]
            all_accessible = True

            for name in targets:
                db_id = _get_db_id(name)
                if not db_id:
                    all_accessible = False
                    msg = f"DB ID not found for '{name}'. Set it in globals/env_config/config['databases']."
                    preflight_results["checks"][name] = {
                        "accessible": False,
                        "database_id": None,
                        "error": msg,
                        "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
                    }
                    print(f"  ✗ {name}: MISSING DB ID")
                    logger.error(msg)
                    continue

                try:
                    resp = _rest_get(session, base, f"/databases/{db_id}", timeout=10)
                    if resp.status_code == 200:
                        data = resp.json()
                        title = None
                        try:
                            title = (data.get("title") or [{}])[0].get("plain_text")
                        except Exception:
                            pass
                        preflight_results["checks"][name] = {
                            "accessible": True,
                            "database_id": db_id,
                            "title": title,
                            "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
                        }
                        print(f"  ✓ {name}: {title or 'accessible'}")
                    else:
                        all_accessible = False
                        err = f"HTTP {resp.status_code}: {resp.text}"
                        preflight_results["checks"][name] = {
                            "accessible": False,
                            "database_id": db_id,
                            "error": err,
                            "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
                        }
                        print(f"  ✗ {name}: NOT ACCESSIBLE ({err})")
                        logger.error("Database '%s' not accessible: %s", name, err)
                except Exception as e:
                    all_accessible = False
                    err = str(e)
                    preflight_results["checks"][name] = {
                        "accessible": False,
                        "database_id": db_id,
                        "error": err,
                        "check_time_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
                    }
                    print(f"  ✗ {name}: NOT ACCESSIBLE ({err})")
                    logger.error("Database '%s' not accessible: %s", name, err)

            preflight_results["overall_status"] = "success" if all_accessible else "failed"

# --- Save preflight results (best effort) ---
preflight_state = {
    "run_id": run_id,
    "timestamp_iso": datetime.now(DEFAULT_TIMEZONE).isoformat(),
    "results": preflight_results,
}
try:
    runs_dir = STATE_DIR / "runs"
    runs_dir.mkdir(parents=True, exist_ok=True)
    preflight_file = runs_dir / f"{run_id}_preflight.json"

    if "atomic_write_json" in globals():
        atomic_write_json(preflight_file, preflight_state, indent=2)
    else:
        with open(preflight_file, "w", encoding="utf-8") as f:
            json.dump(preflight_state, f, indent=2, ensure_ascii=False)

    logger.debug("Preflight results saved to %s", preflight_file.name)
except Exception as e:
    logger.warning("Failed to save preflight results: %s", e)

# --- Summary ---
status = preflight_results["overall_status"]
if status == "success":
    print("\n✓ Cell 08: Preflight checks completed successfully")
elif status == "skipped":
    print("\n✓ Cell 08: Preflight checks skipped")
else:
    print("\n✗ Cell 08: Preflight checks failed")
    if not fail_fast:
        print("  Continuing execution despite failures (fail_fast=False)")

print(f"  - Overall status:  {status}")
print(f"  - Checks performed: {len(preflight_results.get('checks', {}))}")


2026-01-29 04:26:02 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Starting Notion preflight checks (029 optional)
Running Notion preflight checks (lightweight)...
  ✓ Authentication: VERIFIED
  ✓ papers: Literature Database
  ✓ events: EVENTS_DB
  ✓ monitoring_targets: MONITORING_TARGETS_DB
  ✓ monitoring_queue: MONITORING_QUEUE_DB

✓ Cell 08: Preflight checks completed successfully
  - Overall status:  success
  - Checks performed: 5


In [11]:
# ============================================================
# Cell 09 — Exported context helpers for downstream notebooks
# ============================================================
# Overview:
#   Provides convenience functions and context accessors for downstream
#   notebooks (especially 036_daily_orchestrator.ipynb) to retrieve run
#   context, configuration, state paths, and logging without re-implementing
#   initialization logic. Enables clean separation of concerns.
#
# Inputs / Outputs:
#   Inputs:  run_context, config, env_config, state_paths, logger (from prior cells)
#   Outputs: get_run_context(), get_config(), get_state_path(), context export dict
#
# Notes:
#   - All functions are read-only accessors (no mutation)
#   - Designed for safe import/execution from other notebooks
#   - Provides validation helpers for common operations
#   - Includes temporal helpers for date/time calculations
#   - No Notion API calls or I/O operations here
#

# --- Context accessor functions ---

def get_run_context() -> Dict[str, Any]:
    """
    Get the current run context (run_id, timestamps, lookback windows).
    
    Returns:
        Dictionary with run_id, execution_start, run_date, lookback_windows, etc.
    
    Example:
        ctx = get_run_context()
        print(f"Run ID: {ctx['run_id']}")
        print(f"Run date: {ctx['run_date_iso']}")
    """
    return run_context.copy()

def get_config(section: Optional[str] = None) -> Union[Dict[str, Any], Any]:
    """
    Get pipeline configuration, optionally scoped to a specific section.
    
    Args:
        section: Optional dot-notation path (e.g., 'pipeline.cadence')
    
    Returns:
        Full config dict if section=None, otherwise value at path
    
    Example:
        cadence = get_config('pipeline.cadence')
        lookback = get_config('lookback.daily_notes_days')
        full_config = get_config()
    """
    if section is None:
        return config.copy()
    
    return get_nested(config, section)

def get_env_config(key: Optional[str] = None) -> Union[Dict[str, str], str]:
    """
    Get environment configuration (API keys, database IDs).
    
    Args:
        key: Optional environment variable name
    
    Returns:
        Full env_config dict if key=None, otherwise specific value
    
    Example:
        api_key = get_env_config('NOTION_API_KEY')
        daily_db = get_env_config('NOTION_DAILY_DATABASE_ID')
    """
    if key is None:
        return env_config.copy()
    
    if key not in env_config:
        raise KeyError(f"Environment variable '{key}' not found in env_config")
    
    return env_config[key]

def get_state_path(key: str) -> Path:
    """
    Get path to a state file by key.
    
    Args:
        key: State file key (from state_paths dict)
    
    Returns:
        Path object for the state file
    
    Example:
        cursor_path = get_state_path('cursor_daily')
        metadata_path = get_state_path('run_metadata')
    """
    if key not in state_paths:
        raise KeyError(
            f"Unknown state key '{key}', expected one of {list(state_paths.keys())}"
        )
    
    return state_paths[key]

def get_logger_for_notebook(notebook_name: str) -> logging.Logger:
    """
    Get a configured logger instance for a downstream notebook.
    
    Args:
        notebook_name: Identifier for the calling notebook (e.g., '036_daily_orchestrator')
    
    Returns:
        Logger instance with run_id context injection
    
    Example:
        logger = get_logger_for_notebook('036_daily_orchestrator')
        logger.info('Starting daily pipeline')
    """
    return get_logger(notebook_name)

# --- Temporal helper functions ---

def get_lookback_window(database: str) -> Dict[str, Any]:
    """
    Get lookback window configuration for a specific database.
    
    Args:
        database: Database name ('daily_notes' | 'tasks' | 'projects')
    
    Returns:
        Dict with 'days', 'start_date', 'end_date' (date objects)
    
    Example:
        window = get_lookback_window('daily_notes')
        start = window['start_date']  # datetime.date object
        days = window['days']          # int
    """
    if database not in lookback_windows:
        raise ValueError(
            f"Unknown database '{database}', expected one of {list(lookback_windows.keys())}"
        )
    
    return lookback_windows[database].copy()

def get_lookback_start_date(database: str) -> datetime.date:
    """
    Get lookback start date for a database (convenience wrapper).
    
    Args:
        database: Database name ('daily_notes' | 'tasks' | 'projects')
    
    Returns:
        Start date for lookback window
    """
    return get_lookback_window(database)['start_date']

def get_lookback_end_date(database: str) -> datetime.date:
    """
    Get lookback end date for a database (convenience wrapper).
    
    Args:
        database: Database name ('daily_notes' | 'tasks' | 'projects')
    
    Returns:
        End date for lookback window (typically run_date)
    """
    return get_lookback_window(database)['end_date']

def format_date_for_notion(date_obj: datetime.date) -> str:
    """
    Format a date object for Notion API filters (ISO 8601 format).
    
    Args:
        date_obj: Python date object
    
    Returns:
        ISO 8601 formatted date string (YYYY-MM-DD)
    
    Example:
        notion_date = format_date_for_notion(run_date)
        # Returns: '2024-01-15'
    """
    return date_obj.isoformat()

# --- Validation helpers ---

def validate_run_id(candidate: str) -> bool:
    """
    Validate that a string is a properly formatted run_id (UUID4).
    
    Args:
        candidate: String to validate
    
    Returns:
        True if valid run_id format, False otherwise
    
    Example:
        if validate_run_id(some_id):
            logger.info(f"Valid run_id: {some_id}")
    """
    return bool(RUN_ID_PATTERN.match(candidate))

def validate_notion_page_id(page_id: str) -> bool:
    """
    Validate that a string is a properly formatted Notion page ID.
    
    Args:
        page_id: String to validate (with or without hyphens)
    
    Returns:
        True if valid Notion ID format, False otherwise
    """
    normalized = page_id.replace('-', '')
    return bool(NOTION_ID_PATTERN.match(normalized))

def is_within_lookback_window(date_obj: datetime.date, database: str) -> bool:
    """
    Check if a date falls within the lookback window for a database.
    
    Args:
        date_obj: Date to check
        database: Database name ('daily_notes' | 'tasks' | 'projects')
    
    Returns:
        True if date is within [start_date, end_date], False otherwise
    
    Example:
        if is_within_lookback_window(note_date, 'daily_notes'):
            # Process this note
    """
    window = get_lookback_window(database)
    return window['start_date'] <= date_obj <= window['end_date']

# --- Context export dictionary ---
# Provides a single structured export for downstream notebooks

exported_context = {
    'run_id': run_id,
    'run_date_iso': run_date_iso,
    'execution_start_iso': execution_start_iso,
    'config': config,
    'env_config': env_config,
    'state_paths': state_paths,
    'lookback_windows': lookback_windows,
    'preflight_results': preflight_results,
    'state_dir': STATE_DIR,
    'log_file_path': log_file_path,
    'notebook_version': '1.0.0'
}

# --- Convenience function for full context export ---

def get_exported_context() -> Dict[str, Any]:
    """
    Get the complete exported context for downstream notebooks.
    Includes run_id, config, paths, lookback windows, and metadata.
    
    Returns:
        Dictionary with all exported context information
    
    Example:
        # In downstream notebook (e.g., 036_daily_orchestrator.ipynb):
        from config_and_state import get_exported_context
        ctx = get_exported_context()
        logger.info(f"Running with run_id: {ctx['run_id']}")
    """
    return exported_context.copy()

# --- Display helper for interactive inspection ---

def display_context_summary() -> None:
    """
    Display a human-readable summary of the current context.
    Useful for debugging and validation in notebook environments.
    """
    print("="*60)
    print("PIPELINE CONTEXT SUMMARY")
    print("="*60)
    print(f"Run ID:           {run_id}")
    print(f"Run Date:         {run_date_iso}")
    print(f"Execution Start:  {execution_start_iso}")
    print(f"Cadence:          {config['pipeline']['cadence']}")
    print(f"Config Source:    {config_source}")
    print(f"")
    print("Lookback Windows:")
    for db, window in lookback_windows.items():
        print(f"  {db:12s}: {window['days']:2d} days ({window['start_date']} to {window['end_date']})")
    print(f"")
    print("State Management:")
    print(f"  State directory:  {STATE_DIR}")
    print(f"  Log file:         {log_file_path.name}")
    print(f"  State files:      {len(state_paths)}")
    print(f"")
    print("Preflight Status:")
    print(f"  Overall:          {preflight_results['overall_status']}")
    print(f"  Checks run:       {len(preflight_results['checks'])}")
    print("="*60)

# --- Success summary ---
logger.info("Context helpers exported for downstream notebooks")
print("✓ Cell 09: Context helpers exported")
print("  - Accessor functions:")
print("    - get_run_context()")
print("    - get_config(section)")
print("    - get_env_config(key)")
print("    - get_state_path(key)")
print("    - get_logger_for_notebook(name)")
print("  - Temporal helpers:")
print("    - get_lookback_window(database)")
print("    - get_lookback_start_date(database)")
print("    - get_lookback_end_date(database)")
print("    - format_date_for_notion(date)")
print("    - is_within_lookback_window(date, database)")
print("  - Validation helpers:")
print("    - validate_run_id(candidate)")
print("    - validate_notion_page_id(page_id)")
print("  - Export functions:")
print("    - get_exported_context()")
print("    - display_context_summary()")
print("")
print("Exported context available for downstream notebooks")

# --- Optional: Display summary if running interactively ---
# Uncomment to show summary on every run
# display_context_summary()

# --- Export for downstream use ---
# get_run_context(): Access run context dict
# get_config(section): Access configuration values
# get_env_config(key): Access environment variables
# get_state_path(key): Get state file paths
# get_logger_for_notebook(name): Get configured logger
# get_lookback_window(database): Get temporal bounds
# get_exported_context(): Get full context export
# display_context_summary(): Show context summary
# exported_context: Complete context dict for import


2026-01-29 04:26:05 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Context helpers exported for downstream notebooks
✓ Cell 09: Context helpers exported
  - Accessor functions:
    - get_run_context()
    - get_config(section)
    - get_env_config(key)
    - get_state_path(key)
    - get_logger_for_notebook(name)
  - Temporal helpers:
    - get_lookback_window(database)
    - get_lookback_start_date(database)
    - get_lookback_end_date(database)
    - format_date_for_notion(date)
    - is_within_lookback_window(date, database)
  - Validation helpers:
    - validate_run_id(candidate)
    - validate_notion_page_id(page_id)
  - Export functions:
    - get_exported_context()
    - display_context_summary()

Exported context available for downstream notebooks


In [12]:
# ============================================================
# Cell 10 — Configuration summary and sanity checks
# ============================================================
# Overview:
#   Performs final validation of the complete configuration and run
#   context, checking for logical consistency, valid ranges, and
#   potential misconfigurations. Provides a summary report suitable
#   for audit logs and debugging.
#
# Inputs / Outputs:
#   Inputs:  config, env_config, run_context, lookback_windows, preflight_results
#   Outputs: validation_report dict, warnings list, critical issues list
#
# Notes:
#   - Validates cross-section configuration consistency
#   - Checks for common misconfigurations and edge cases
#   - Does not modify configuration, only validates and reports
#   - Logs warnings for suspicious but non-fatal issues
#   - Raises errors only for critical misconfigurations
#

# --- Validation checks to perform ---
validation_warnings = []
validation_errors = []

# --- Check 1: Lookback window consistency ---
for db_name, window in lookback_windows.items():
    days = window['days']
    
    # Check for unreasonable lookback periods
    if days > 365:
        validation_warnings.append(
            f"Lookback window for {db_name} is very long ({days} days). "
            f"This may cause performance issues."
        )
    
    # Verify date logic
    if window['start_date'] >= window['end_date']:
        validation_errors.append(
            f"Invalid lookback window for {db_name}: "
            f"start_date ({window['start_date']}) >= end_date ({window['end_date']})"
        )

# --- Check 2: Runtime and threshold limits ---
max_runtime = config['pipeline']['max_runtime_minutes']
if max_runtime < 5:
    validation_warnings.append(
        f"Max runtime ({max_runtime} min) may be too short for full pipeline execution"
    )

max_items = config['thresholds']['max_items_per_query']
if max_items > 100:
    validation_warnings.append(
        f"max_items_per_query ({max_items}) exceeds Notion API pagination limit (100)"
    )

# --- Check 3: State directory writability ---
if not STATE_DIR.exists() or not os.access(STATE_DIR, os.W_OK):
    validation_errors.append(
        f"State directory {STATE_DIR} is not writable"
    )

# --- Check 4: Preflight status consistency ---
if config['pipeline']['enable_preflight_checks']:
    if preflight_results['overall_status'] == 'failed' and not config['pipeline']['fail_fast']:
        validation_warnings.append(
            "Preflight checks failed but fail_fast=False. "
            "Pipeline will proceed despite connectivity issues."
        )

# --- Check 5: Logging configuration ---
if not config['logging']['log_to_file'] and not config['logging']['log_to_console']:
    validation_errors.append(
        "Both file and console logging are disabled. No logs will be captured."
    )

# --- Build validation report ---
validation_report = {
    'run_id': run_id,
    'timestamp_iso': datetime.now(DEFAULT_TIMEZONE).isoformat(),
    'warnings': validation_warnings,
    'errors': validation_errors,
    'checks_performed': 5,
    'status': 'failed' if validation_errors else ('warnings' if validation_warnings else 'passed')
}

# --- Log results ---
logger.info(f"Configuration validation complete: {validation_report['status']}")

if validation_warnings:
    logger.warning(f"Found {len(validation_warnings)} configuration warnings:")
    for warning in validation_warnings:
        logger.warning(f"  - {warning}")

if validation_errors:
    logger.error(f"Found {len(validation_errors)} configuration errors:")
    for error in validation_errors:
        logger.error(f"  - {error}")

# --- Handle critical errors ---
if validation_errors:
    error_msg = "Critical configuration errors detected:\n  " + "\n  ".join(validation_errors)
    raise RuntimeError(error_msg)

# --- Display summary ---
print("\n✓ Cell 10: Configuration validation complete")
print(f"  - Status: {validation_report['status'].upper()}")
print(f"  - Warnings: {len(validation_warnings)}")
print(f"  - Errors: {len(validation_errors)}")

if validation_warnings:
    print("\n  ⚠ Warnings detected:")
    for warning in validation_warnings:
        print(f"    - {warning}")

# --- Save validation report to state ---
try:
    validation_file = STATE_DIR / 'runs' / f"{run_id}_validation.json"
    with open(validation_file, 'w', encoding='utf-8') as f:
        json.dump(validation_report, f, indent=2, ensure_ascii=False)
    logger.debug(f"Validation report saved to {validation_file.name}")
except Exception as e:
    logger.warning(f"Failed to save validation report: {e}")

# --- Export for downstream use ---
# validation_report: Dict with validation results
# validation_warnings: List of non-critical issues
# validation_errors: List of critical issues (empty if passed)


2026-01-29 04:26:06 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Configuration validation complete: passed

✓ Cell 10: Configuration validation complete
  - Status: PASSED
  - Warnings: 0
  - Errors: 0


In [13]:
# ============================================================
# Cell 11 — RunContext manager for encapsulated execution
# ============================================================
# Overview:
#   Provides a context manager class for encapsulating pipeline execution
#   with automatic state management, error handling, status tracking, and
#   cleanup. Ensures consistent lifecycle management across pipeline runs
#   and enables safe rollback on errors.
#
# Inputs / Outputs:
#   Inputs:  run_context, config, state_paths, logger (from prior cells)
#   Outputs: RunContext class, context manager instance factory
#
# Notes:
#   - Implements __enter__ and __exit__ for context manager protocol
#   - Automatically saves run state on success or error
#   - Handles graceful cleanup and state finalization
#   - Provides status tracking (initialized, running, success, failed)
#   - Exports execution summary on exit
#   - Thread-safe state updates (atomic writes)
#

# --- RunContext class definition ---

class RunContext:
    """
    Context manager for pipeline execution with automatic state management.
    
    Usage:
        with RunContext() as ctx:
            # Pipeline execution code
            ctx.log_progress('Processing daily notes')
            ctx.increment_counter('pages_processed', 10)
    
    Features:
        - Automatic state initialization and finalization
        - Error handling with rollback support
        - Progress tracking and counters
        - Execution timing and performance metrics
        - State persistence on success or failure
    """
    
    def __init__(self, name: str = 'pipeline'):
        """
        Initialize RunContext.
        
        Args:
            name: Human-readable name for this execution context
        """
        self.name = name
        self.run_id = run_id
        self.start_time = None
        self.end_time = None
        self.status = 'initialized'
        self.error = None
        self.counters = {}
        self.progress_log = []
        self.metadata = {}
        
        # Reference to global logger
        self.logger = get_logger(f'RunContext.{name}')
    
    def __enter__(self):
        """
        Enter context: mark execution start and log initialization.
        """
        self.start_time = datetime.now(DEFAULT_TIMEZONE)
        self.status = 'running'
        
        self.logger.info(f"RunContext '{self.name}' started")
        self.logger.debug(f"Start time: {self.start_time.isoformat()}")
        
        # Update run metadata state
        try:
            current_metadata = load_state('run_metadata', default={})
            current_metadata['status'] = 'running'
            current_metadata['execution_start_iso'] = self.start_time.isoformat()
            current_metadata['context_name'] = self.name
            save_state('run_metadata', current_metadata, backup=False)
        except Exception as e:
            self.logger.warning(f"Failed to update run metadata on enter: {e}")
        
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        Exit context: finalize state, log summary, handle errors.
        
        Args:
            exc_type: Exception type (if raised)
            exc_val: Exception value (if raised)
            exc_tb: Exception traceback (if raised)
        
        Returns:
            False to propagate exceptions, True to suppress
        """
        self.end_time = datetime.now(DEFAULT_TIMEZONE)
        
        # Determine final status
        if exc_type is not None:
            self.status = 'failed'
            self.error = {
                'type': exc_type.__name__,
                'message': str(exc_val),
                'timestamp_iso': self.end_time.isoformat()
            }
            self.logger.error(f"RunContext '{self.name}' failed: {exc_val}")
        else:
            self.status = 'success'
            self.logger.info(f"RunContext '{self.name}' completed successfully")
        
        # Calculate execution duration
        duration_seconds = (self.end_time - self.start_time).total_seconds()
        
        # Build execution summary
        summary = {
            'run_id': self.run_id,
            'context_name': self.name,
            'status': self.status,
            'start_time_iso': self.start_time.isoformat(),
            'end_time_iso': self.end_time.isoformat(),
            'duration_seconds': duration_seconds,
            'counters': self.counters.copy(),
            'progress_log': self.progress_log.copy(),
            'metadata': self.metadata.copy(),
            'error': self.error
        }
        
        self.logger.debug(f"Duration: {duration_seconds:.2f}s")
        self.logger.debug(f"Counters: {self.counters}")
        
        # Save execution summary to state
        try:
            save_state('run_summary', summary, backup=False)
            self.logger.debug("Execution summary saved to state")
        except Exception as e:
            self.logger.error(f"Failed to save execution summary: {e}")
        
        # Update run metadata with final status
        try:
            current_metadata = load_state('run_metadata', default={})
            current_metadata['status'] = self.status
            current_metadata['execution_end_iso'] = self.end_time.isoformat()
            current_metadata['duration_seconds'] = duration_seconds
            if self.error:
                current_metadata['error'] = self.error
            save_state('run_metadata', current_metadata, backup=False)
        except Exception as e:
            self.logger.error(f"Failed to update run metadata on exit: {e}")
        
        # Backup state on error if configured
        if self.status == 'failed' and config['state']['backup_on_error']:
            try:
                self._backup_state_files()
                self.logger.info("State files backed up due to execution failure")
            except Exception as e:
                self.logger.error(f"Failed to backup state files: {e}")
        
        # Log final summary
        self.logger.info(
            f"RunContext '{self.name}' finalized: "
            f"status={self.status}, duration={duration_seconds:.2f}s"
        )
        
        # Do not suppress exceptions (return False or None)
        return False
    
    def log_progress(self, message: str, level: str = 'info') -> None:
        """
        Log a progress message and add to progress log.
        
        Args:
            message: Progress message
            level: Log level (info | debug | warning | error)
        """
        timestamp = datetime.now(DEFAULT_TIMEZONE).isoformat()
        
        # Add to internal progress log
        self.progress_log.append({
            'timestamp_iso': timestamp,
            'message': message,
            'level': level
        })
        
        # Log to logger
        log_method = getattr(self.logger, level, self.logger.info)
        log_method(f"[Progress] {message}")
    
    def increment_counter(self, name: str, value: int = 1) -> int:
        """
        Increment a named counter.
        
        Args:
            name: Counter name
            value: Amount to increment (default: 1)
        
        Returns:
            New counter value
        """
        if name not in self.counters:
            self.counters[name] = 0
        
        self.counters[name] += value
        
        self.logger.debug(f"Counter '{name}' incremented by {value} -> {self.counters[name]}")
        
        return self.counters[name]
    
    def set_counter(self, name: str, value: int) -> None:
        """
        Set a named counter to a specific value.
        
        Args:
            name: Counter name
            value: Counter value
        """
        self.counters[name] = value
        self.logger.debug(f"Counter '{name}' set to {value}")
    
    def get_counter(self, name: str, default: int = 0) -> int:
        """
        Get current value of a named counter.
        
        Args:
            name: Counter name
            default: Default value if counter doesn't exist
        
        Returns:
            Counter value
        """
        return self.counters.get(name, default)
    
    def set_metadata(self, key: str, value: Any) -> None:
        """
        Set a metadata key-value pair.
        
        Args:
            key: Metadata key
            value: Metadata value (must be JSON-serializable)
        """
        self.metadata[key] = value
        self.logger.debug(f"Metadata '{key}' set")
    
    def get_metadata(self, key: str, default: Any = None) -> Any:
        """
        Get a metadata value.
        
        Args:
            key: Metadata key
            default: Default value if key doesn't exist
        
        Returns:
            Metadata value
        """
        return self.metadata.get(key, default)
    
    def _backup_state_files(self) -> None:
        """
        Create backups of critical state files.
        Internal method called on error if backup_on_error is enabled.
        """
        critical_state_keys = [
            'cursor_daily', 'cursor_tasks', 'cursor_projects',
            'processed_daily', 'processed_tasks', 'processed_projects'
        ]
        
        for key in critical_state_keys:
            try:
                path = state_paths[key]
                if path.exists():
                    backup_path = backup_state_file(path)
                    if backup_path:
                        self.logger.debug(f"Backed up {key}: {backup_path.name}")
            except Exception as e:
                self.logger.warning(f"Failed to backup {key}: {e}")
    
    def get_elapsed_seconds(self) -> float:
        """
        Get elapsed time since context start.
        
        Returns:
            Elapsed seconds (float)
        """
        if self.start_time is None:
            return 0.0
        
        current_time = datetime.now(DEFAULT_TIMEZONE)
        return (current_time - self.start_time).total_seconds()
    
    def check_timeout(self, max_runtime_minutes: Optional[int] = None) -> bool:
        """
        Check if execution has exceeded maximum runtime.
        
        Args:
            max_runtime_minutes: Override for config value
        
        Returns:
            True if timeout exceeded, False otherwise
        """
        max_minutes = max_runtime_minutes or config['pipeline']['max_runtime_minutes']
        elapsed_minutes = self.get_elapsed_seconds() / 60.0
        
        if elapsed_minutes > max_minutes:
            self.logger.warning(
                f"Execution timeout exceeded: {elapsed_minutes:.1f} min > {max_minutes} min"
            )
            return True
        
        return False

# --- Factory function for creating RunContext instances ---

def create_run_context(name: str = 'pipeline') -> RunContext:
    """
    Factory function for creating RunContext instances.
    
    Args:
        name: Human-readable name for the context
    
    Returns:
        New RunContext instance
    
    Example:
        with create_run_context('daily_pipeline') as ctx:
            ctx.log_progress('Starting daily processing')
            # ... pipeline code ...
            ctx.increment_counter('pages_processed', 42)
    """
    return RunContext(name=name)

# --- Success summary ---
logger.info("RunContext manager defined")
print("✓ Cell 11: RunContext manager implemented")
print("  - Class: RunContext")
print("  - Features:")
print("    - Automatic state initialization and finalization")
print("    - Progress logging and counter tracking")
print("    - Error handling with state backup")
print("    - Execution timing and timeout checks")
print("  - Factory: create_run_context(name)")
print("  ")
print("  Usage example:")
print("    with create_run_context('my_pipeline') as ctx:")
print("        ctx.log_progress('Step 1 complete')")
print("        ctx.increment_counter('items_processed')")
print("        ctx.set_metadata('source', 'daily_notes')")

# --- Export for downstream use ---
# RunContext: Context manager class for pipeline execution
# create_run_context(name): Factory function for creating instances


2026-01-29 04:26:07 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | RunContext manager defined
✓ Cell 11: RunContext manager implemented
  - Class: RunContext
  - Features:
    - Automatic state initialization and finalization
    - Progress logging and counter tracking
    - Error handling with state backup
    - Execution timing and timeout checks
  - Factory: create_run_context(name)
  
  Usage example:
    with create_run_context('my_pipeline') as ctx:
        ctx.log_progress('Step 1 complete')
        ctx.increment_counter('items_processed')
        ctx.set_metadata('source', 'daily_notes')


In [14]:
# ============================================================
# Cell 12 — Self-test and validation suite
# ============================================================
# Overview:
#   Executes a comprehensive self-test suite to validate that all
#   configuration, state management, logging, and context helpers
#   are functioning correctly. Ensures the notebook is ready for
#   production use by downstream orchestrators.
#
# Inputs / Outputs:
#   Inputs:  All prior cell outputs (config, state, context, logger)
#   Outputs: test_results dict with pass/fail status per test
#
# Notes:
#   - Tests are defensive and isolated (no side effects)
#   - Failures are logged but do not block notebook completion
#   - Provides clear diagnostics for debugging misconfigurations
#   - Skips tests that require external dependencies (e.g., 029)
#

# --- Test suite definition ---
test_results = {
    'run_id': run_id,
    'timestamp_iso': datetime.now(DEFAULT_TIMEZONE).isoformat(),
    'tests': {},
    'summary': {'passed': 0, 'failed': 0, 'skipped': 0}
}

def run_test(name: str, test_func, skip: bool = False):
    """Execute a single test and record result."""
    if skip:
        test_results['tests'][name] = {'status': 'skipped', 'error': None}
        test_results['summary']['skipped'] += 1
        return
    
    try:
        test_func()
        test_results['tests'][name] = {'status': 'passed', 'error': None}
        test_results['summary']['passed'] += 1
    except Exception as e:
        test_results['tests'][name] = {'status': 'failed', 'error': str(e)}
        test_results['summary']['failed'] += 1
        logger.error(f"Test '{name}' failed: {e}")

# --- Test 1: Run ID validation ---
def test_run_id():
    assert RUN_ID_PATTERN.match(run_id), f"Invalid run_id format: {run_id}"
    assert run_id == run_context['run_id'], "run_id mismatch in context"

run_test('run_id_format', test_run_id)

# --- Test 2: Config accessibility ---
def test_config_access():
    assert get_config('pipeline.cadence') in ['daily', 'hourly', 'manual']
    assert isinstance(get_config('lookback.daily_notes_days'), int)
    assert get_config() is not None

run_test('config_access', test_config_access)

# --- Test 3: State directory structure ---
def test_state_directory():
    assert STATE_DIR.exists(), f"State directory missing: {STATE_DIR}"
    for subdir in STATE_SUBDIRS:
        assert (STATE_DIR / subdir).exists(), f"Missing subdir: {subdir}"

run_test('state_directory', test_state_directory)

# --- Test 4: State persistence roundtrip ---
def test_state_persistence():
    test_data = {'test_key': 'test_value', 'timestamp': datetime.now(DEFAULT_TIMEZONE).isoformat()}
    test_key = 'run_metadata'  # Use existing key
    assert save_state(test_key, test_data, backup=False)
    loaded = load_state(test_key)
    assert loaded['test_key'] == 'test_value', "State roundtrip failed"

run_test('state_persistence', test_state_persistence)

# --- Test 5: Logger functionality ---
def test_logger():
    test_logger = get_logger_for_notebook('self_test')
    assert test_logger is not None
    test_logger.info("Test log message")
    assert log_file_path.exists(), f"Log file not created: {log_file_path}"

run_test('logger', test_logger)

# --- Test 6: Lookback window calculations ---
def test_lookback_windows():
    for db in ['daily_notes', 'tasks', 'projects']:
        window = get_lookback_window(db)
        assert window['start_date'] < window['end_date'], f"Invalid window for {db}"
        assert window['days'] > 0, f"Invalid lookback days for {db}"

run_test('lookback_windows', test_lookback_windows)

# --- Test 7: Context export completeness ---
def test_context_export():
    ctx = get_exported_context()
    required_keys = ['run_id', 'config', 'env_config', 'state_paths', 'lookback_windows']
    for key in required_keys:
        assert key in ctx, f"Missing key in exported context: {key}"

run_test('context_export', test_context_export)

# --- Test 8: RunContext manager ---
def test_run_context_manager():
    with create_run_context('test_context') as ctx:
        ctx.log_progress('Test step')
        ctx.increment_counter('test_counter', 5)
        assert ctx.get_counter('test_counter') == 5
        assert ctx.status == 'running'

run_test('run_context_manager', test_run_context_manager)

# --- Summary ---
logger.info(f"Self-test complete: {test_results['summary']}")
print("\n✓ Cell 12: Self-test suite executed")
print(f"  - Passed:  {test_results['summary']['passed']}")
print(f"  - Failed:  {test_results['summary']['failed']}")
print(f"  - Skipped: {test_results['summary']['skipped']}")

if test_results['summary']['failed'] > 0:
    print("\n  ⚠ Some tests failed:")
    for name, result in test_results['tests'].items():
        if result['status'] == 'failed':
            print(f"    - {name}: {result['error']}")

# Export test_results for inspection


2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Test log message
2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | RunContext 'test_context' started
2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | [Progress] Test step
2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | RunContext 'test_context' completed successfully
2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | RunContext 'test_context' finalized: status=success, duration=0.01s
2026-01-29 04:26:08 | INFO     | 775d9a47-7a9a-4ddf-bf9c-e82c22063cd9 | Self-test complete: {'passed': 8, 'failed': 0, 'skipped': 0}

✓ Cell 12: Self-test suite executed
  - Passed:  8
  - Failed:  0
  - Skipped: 0
